In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from scipy.stats import norm
from scipy.integrate import simpson
import matplotlib.pyplot as plt
from pathlib import Path

# Settings
TAU_DAYS = 30
MIND2E = 14
MAXD2E = 60
EPSILON = 1e-8
ROOT = Path('')


# 1.0 Theoretical Background and Introduction

This project builds on the methodology introduced by **Bakshi, Kapadia, and Madan (2003)**, which provides a  way to extract the **risk-neutral moments** of an asset's return distribution directly from **market option prices**. Under the risk-neutral measure, option prices cointain the market’s expectations about future returns, including not only the mean and variance, but also higher-order features like **skewness** and **kurtosis**.

$$
\text{Risk-neutral moments:} \quad \mu(t,\tau), \quad \sigma^2(t,\tau), \quad \text{SKEW}(t,\tau), \quad \text{KURT}(t,\tau)
$$

These quantities are estimated from integrals over out-of-the-money (OTM) call and put option prices, weighted in specific ways as shown in the BKM framework. The idea is that option prices reflect the **risk-neutral probability distribution** of future returns, and this distribution can be characterized by its moments.

### Why compute risk-neutral moments?

- **$\mu(t,\tau)$** captures the drift implied under the risk-neutral measure.
- **$\sigma^2(t,\tau)$** measures the market’s expectation of future variance (related to implied volatility).
- **Skewness** reflects the market’s perception of downside risk or left-tail events.
- **Kurtosis** captures the weight of extreme outcomes — tail risk — beyond what is implied by normal distributions.

Understanding these moments is useful for:

- **Risk management**: Tail risks are not visible in volatility alone.
- **Derivative pricing**: Models beyond Black-Scholes require knowledge of higher moments.
- **Market sentiment analysis**: Persistent negative skewness, for example, reflects crash risk pricing.

This theoretical setup allows us to investigate how expectations of risk and asymmetry differ across **individual stocks** and the **market index**, and how much of individual skewness is driven by **systematic** versus **idiosyncratic** components.


---
---
### 1.1 Black-Scholes Option Pricing Formula 

The function `bs_price(...)` implements the **Black-Scholes formula**, which is a fundamental model in financial economics for pricing European-style options. It provides a solution for the price of a call or put option, assuming that markets are efficient.

### Mathematical Formulation

The Black-Scholes prices for a European call (C) and put (P) are given by:

- **Call option**:
$$
C = e^{-q \tau} S \Phi(d_1) - e^{-r \tau} K \Phi(d_2)
$$

- **Put option**:
$$
P = e^{-r \tau} K \Phi(-d_2) - e^{-q \tau} S \Phi(-d_1)
$$

where

$$
d_1 = \frac{\ln(S/K) + (r - q + \frac{1}{2} \sigma^2)\tau}{\sigma \sqrt{\tau}}, \quad
d_2 = d_1 - \sigma \sqrt{\tau}
$$

Here:
- $S$ is the current underlying price  
- $K$ is the strike price  
- $r$ is the continuously compounded risk-free interest rate  
- $q$ is the continuous dividend yield  
- $\sigma$ is the volatility of the underlying asset  
- $\tau$ is the time to maturity in years  
- $\Phi(\cdot)$ is the cumulative distribution function of the standard normal distribution  

### Intuition and Usage

The Black-Scholes model calculates the **fair value** of an option under the assumption of no arbitrage. The two components in each formula represent:

- The **expected benefit of holding the option** 
- The **cost of carrying** that right, discounted back to present value

The function `bs_price()` is used in our project to:
- Estimate theoretical option prices based on observed implied volatilities
- Derive implied forward prices using put-call parity
- Replace actual option prices when needed to ensure consistency in integral-based moment calculations (as in the BKM framework)

While BKM (2003) ultimately relies on market prices rather than model outputs, Black-Scholes remains useful for **interpolation**, **forward estimation**, and **sanity-checking** price relationships.


In [ ]:
def bs_price(S, K, r, q, vol, tau, flag):
    if vol <= 0 or tau <= 0:
        return 0.0
    d1 = (np.log(S / K) + (r - q + 0.5 * vol ** 2) * tau) / (vol * np.sqrt(tau))
    d2 = d1 - vol * np.sqrt(tau)
    if flag == 'C':
        return np.exp(-q * tau) * S * norm.cdf(d1) - np.exp(-r * tau) * K * norm.cdf(d2)
    else:
        return np.exp(-r * tau) * K * norm.cdf(-d2) - np.exp(-q * tau) * S * norm.cdf(-d1)


### 1.2 Risk-Free Rate Preparation

To compute risk-neutral moments, we require the continuously compounded risk-free interest rate $r(t)$ for the given maturity horizon ($\tau = 30$ days).

We import the daily risk-free yield curve data from the file `riskfree_30d_2023-02.csv`, which contains the 30-day U.S. Treasury yield expressed in annualized percentage terms. We then convert the annualized yield to a **continuously compounded** rate:

$$
r_{\text{cc}}(t) = \frac{\text{yld\_pct\_annual}(t)}{100}
$$

This column `rf_cc` is used throughout the project to discount cash flows and compute forward prices as required by the BKM (2003) methodology.


In [ ]:
rf = pd.read_csv(ROOT / 'riskfree_30d_2023-02.csv', parse_dates=['date'])
rf['rf_cc'] = rf['yld_pct_annual'] / 100


In [ ]:
rf.head()


### 1.3 Preprocessing Option Data

To ensure consistency and usability of the option datasets, we define a preprocessing function that:

1. Renames key columns to standardized names

2. Converts the `date` column to `datetime` format and ensures `d2e` is an integer.

3. Merges the option data with the corresponding **risk-free rate** (`rf_cc`) by matching on the date.

Finally, this function is applied to load:
- `spx`: Implied volatility surface data for the S&P 500 index
- `const`: Implied volatility data for all S&P 500 constituents


In [ ]:
def preprocess(df):
    df = df.copy()
    df.rename(columns={
        'loctimestamp': 'date',
        'putcall': 'cp_flag',
        'implVol': 'iv',
        'daystomaturity': 'd2e',
        'Symbol': 'ticker',
        'underlyingprice': 'underlying_price'
    }, inplace=True)
    df['date'] = pd.to_datetime(df['date'])
    df['d2e'] = df['d2e'].astype(int)
    return df.merge(rf[['date', 'rf_cc']], on='date', how='left')

spx = preprocess(pq.read_table(ROOT / 'spx_ivs_2023-02.parquet').to_pandas())
const = preprocess(pq.read_table(ROOT / 'sp500_merged_ivs_2023-02.parquet').to_pandas())


In [ ]:

print("SPX:")
display(spx.head())

print("CONST:")
display(const.head())


## 1.4 Estimating Risk-Neutral Moments from Option Prices (BKM, 2003)

In this part of the project, we estimate the **first four risk-neutral moments** of the return distribution of the underlying asset (index or stock) based on the option prices, using the framework of Bakshi, Kapadia, and Madan (2003). These are:

- $\mu(t, \tau)$: Risk-neutral mean (expected return)
- $\sigma^2(t, \tau)$: Risk-neutral variance
- $\text{SKEW}(t, \tau)$: Risk-neutral skewness (3rd standardized moment)
- $\text{KURT}(t, \tau)$: Risk-neutral kurtosis (4th standardized moment)

The goal is to extract the **shape** of the return distribution implied by option markets — not under the physical (real-world) probability measure, but under the **risk-neutral measure**, which reflects the pricing of risk in financial markets.

---

### Why is this important?

- **Option prices contain forward-looking information** about the market's expectations of future risk, including not just volatility but also asymmetry (skewness) and tail risk (kurtosis).
- These moments are **model-free** in the sense that they are not derived from assumptions about return dynamics, but directly from observed option prices.
- Extracting these moments helps understand **markets**, **volatility risk premia**, and **systematic vs. idiosyncratic risk** — all of which are important in asset pricing and risk management.

---

### Methodology

We assume that the cross-section of European option prices contains all the information needed to reconstruct the **risk-neutral density** of the asset's return over horizon $\tau$. From this, we can extract the moments using specific weighted integrals over OTM call and put prices.

Let $S(t)$ be the spot price at time $t$, and let $K$ denote strike price. Then we compute:

#### 1. Variance-related integral $V(t, \tau)$:

$$
V(t, \tau) = \int_{S(t)}^\infty \frac{2}{K^2} C(t, \tau; K) \, dK + \int_0^{S(t)} \frac{2}{K^2} P(t, \tau; K) \, dK
$$

This captures the **second central moment**, i.e., the expected squared return under the risk-neutral measure.

#### 2. Skewness-related integral $W(t, \tau)$:

$$
W(t, \tau) = \int_{S(t)}^\infty \frac{6 \ln \left( \frac{K}{S(t)} \right) - 3 \ln^2 \left( \frac{K}{S(t)} \right)}{K^2} C(t, \tau; K) \, dK 
- \int_0^{S(t)} \frac{6 \ln \left( \frac{S(t)}{K} \right) + 3 \ln^2 \left( \frac{S(t)}{K} \right)}{K^2} P(t, \tau; K) \, dK
$$

This approximates the **third central moment** (asymmetry) of the return distribution.

#### 3. Kurtosis-related integral $X(t, \tau)$:

$$
X(t, \tau) = \int_{S(t)}^\infty \frac{12 \ln^2 \left( \frac{K}{S(t)} \right) - 4 \ln^3 \left( \frac{K}{S(t)} \right)}{K^2} C(t, \tau; K) \, dK 
+ \int_0^{S(t)} \frac{12 \ln^2 \left( \frac{S(t)}{K} \right) + 4 \ln^3 \left( \frac{S(t)}{K} \right)}{K^2} P(t, \tau; K) \, dK
$$

This reflects the **fourth central moment**, capturing the "tailedness" or likelihood of extreme events.

---

### Final Computation of Moments

With the quantities $V(t, \tau)$, $W(t, \tau)$, and $X(t, \tau)$ in hand, and letting $e^{r\tau}$ be the risk-free discount factor over $\tau$ years, we compute:

- Risk-neutral mean (expected return):

$$
\mu(t, \tau) = e^{r\tau} - 1 - \frac{e^{r\tau}}{2} V(t, \tau) - \frac{e^{r\tau}}{6} W(t, \tau) - \frac{e^{r\tau}}{24} X(t, \tau)
$$

- Risk-neutral variance:

$$
\sigma^2(t, \tau) = e^{r\tau} \cdot V(t, \tau) - \mu(t, \tau)^2
$$

- Risk-neutral skewness:

$$
\text{SKEW}(t, \tau) = \frac{e^{r\tau} W(t, \tau) - 3 \mu(t, \tau) e^{r\tau} V(t, \tau) + 2 \mu(t, \tau)^3}{\sigma(t, \tau)^3}
$$

- Risk-neutral kurtosis:

$$
\text{KURT}(t, \tau) = \frac{e^{r\tau} X(t, \tau) - 4 \mu(t, \tau) e^{r\tau} W(t, \tau) + 6 \mu(t, \tau)^2 e^{r\tau} V(t, \tau) - 3 \mu(t, \tau)^4}{\sigma(t, \tau)^4}
$$

---

### Interpretation

This approach allows us to reconstruct the **entire shape of the return distribution** as perceived by the market. It is especially powerful because:

- It works across **stocks and indices**, enabling comparison of individual vs. aggregate risk.
- It gives insight into **systematic risk** (market-wide skew) versus **idiosyncratic skew** (firm-specific risk).
- It reveals how the market **prices tail risk**, which is not visible in standard volatility measures.



In [ ]:
def daily_moments(day_df, r, tau_yrs):
    def calc_forward(df, r, tau):
        S = df['underlying_price'].iloc[0]
        calls = df[df['cp_flag'] == 'C']
        puts = df[df['cp_flag'] == 'P']

        # Gemeinsame Strikes, damit Put-Call-Parität möglich ist
        common_strikes = np.intersect1d(calls['strike'], puts['strike'])
        if len(common_strikes) == 0:
            return S * np.exp(r * tau), S, np.nan, np.nan, np.nan  # fallback

        # Bestimme Strike mit kleinstem |C - P|
        min_diff = np.inf
        best_K = None
        for K in common_strikes:
            C_row = calls[calls['strike'] == K]
            P_row = puts[puts['strike'] == K]
            if C_row.empty or P_row.empty:
                continue
            C_mkt = C_row['implPrice'].values[0]
            P_mkt = P_row['implPrice'].values[0]
            diff = abs(C_mkt - P_mkt)
            if diff < min_diff:
                min_diff = diff
                best_K = K
                C, P = C_mkt, P_mkt

        if best_K is None:
            return S * np.exp(r * tau), S, np.nan, np.nan, np.nan

        F = best_K + np.exp(r * tau) * (C - P)
        return F, S, best_K, C, P

    # Berechne Forward F aus Marktpreisen
    F, S, K_star, C, P = calc_forward(day_df, r, tau_yrs)

    calls = day_df[(day_df['cp_flag'] == 'C') & (day_df['strike'] >= S)].copy()
    puts = day_df[(day_df['cp_flag'] == 'P') & (day_df['strike'] <= S)].copy()

    if len(calls) < 5 or len(puts) < 5:
        return np.nan, np.nan, np.nan, np.nan, F, S, K_star, C, P

    # Verwende Forward-Modell für Preisbewertung (q = r - log(F/S)/tau)
    q = r - np.log(F / S) / tau_yrs

    for df_opt, flag in ((calls, 'C'), (puts, 'P')):
        df_opt['price'] = [bs_price(S, K, r, q, vol, tau_yrs, flag)
                           for K, vol in zip(df_opt['strike'], df_opt['iv'])]

    def trap(df, weight):
        df = df.sort_values('strike')
        K = df['strike'].values
        return simpson(df['price'].values * weight(K), K)

    V = trap(calls, lambda K: 2 / K**2) + trap(puts, lambda K: 2 / K**2)
    W = trap(calls, lambda K: (6*np.log(K/S) - 3*np.log(K/S)**2) / K**2) \
      - trap(puts, lambda K: (6*np.log(S/K) + 3*np.log(S/K)**2) / K**2)
    X = trap(calls, lambda K: (12*np.log(K/S)**2 - 4*np.log(K/S)**3) / K**2) \
      + trap(puts, lambda K: (12*np.log(S/K)**2 + 4*np.log(S/K)**3) / K**2)

    ert = np.exp(r * tau_yrs)
    mu = ert - 1 - (ert / 2) * V - (ert / 6) * W - (ert / 24) * X
    sigma2 = ert * V - mu**2
    if sigma2 < EPSILON:
        return mu, EPSILON, np.nan, np.nan, F, S, K_star, C, P

    skew = (ert * W - 3 * mu * ert * V + 2 * mu**3) / (sigma2 ** 1.5)
    kurt = (ert * X - 4 * mu * ert * W + 6 * mu**2 * ert * V - 3 * mu**4) / (sigma2 ** 2)

    return mu, sigma2, skew, kurt, F, S, K_star, C, P


## 1.5 Computing Daily Risk-Neutral Moments

This loop computes daily **risk-neutral moments** (mean $\mu$, variance $\sigma^2$, skewness, kurtosis) for each stock and for the SPX index.

- For each ticker and trading day, the code selects option data with time-to-expiry ($d2e$) close to a target horizon.
- It computes the annualized time to expiry $\tau$ and uses the corresponding risk-free rate $r$.
- The function `daily_moments` is called to extract the implied return distribution moments from the option prices.
- Results are stored in a list and converted to a sorted DataFrame `mom`.

This forms the basis for analyzing option-implied expectations and higher-order risks across assets and time.


In [ ]:
records = []

for ticker, subset in [('SPX', spx)] + list(const.groupby('ticker')):
    for date, day in subset.groupby('date'):
        choices = day[(day['d2e'] >= MIND2E) & (day['d2e'] <= MAXD2E)]
        if choices.empty:
            continue
        idx = choices['d2e'].sub(TAU_DAYS).abs().idxmin()
        tau_yrs = choices.loc[idx, 'd2e'] / 365.0
        rf_cc = choices.loc[idx, 'rf_cc']
        tau_df = choices[choices['d2e'] == choices.loc[idx, 'd2e']]
        mu, var, skew, kurt, F, S, K_star, C, P = daily_moments(
            day_df=tau_df,
            r=rf_cc, tau_yrs=tau_yrs
        )
        records.append(dict(
            date=date, ticker=ticker, mu=mu, var=var, skew=skew, kurt=kurt,
            F=F, S=S, 
        ))

mom = pd.DataFrame.from_records(records).sort_values(['ticker', 'date'])


---
---
### Side Note: Forward Spread Analysis

This plot shows the difference between the forward price and the spot price of the SPX index over time. 

By looking at this spread, we can observe the cost of carrying the asset forward, which reflects interest rates and the time value of money. 

In [ ]:

spx_df = mom[mom['ticker'] == 'SPX'].copy()
spx_df['F_minus_S'] = spx_df['F'] - spx_df['S']

plt.figure(figsize=(10, 5))
plt.plot(spx_df['date'], spx_df['F_minus_S'], marker='o', linestyle='-')
plt.title('SPX Forward Price minus Spot Price (F - S)')
plt.xlabel('Date')
plt.ylabel('Forward Spread (F - S)')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


#### Why is $F > S$ and not $F = S$?

The no-arbitrage forward price  is:

$$
F = S \cdot e^{(r - q)\tau}
$$

If you assume $q = 0$ (no dividends), then:

$$
F = S \cdot e^{r \tau} > S
$$

This reflects the time value of money: you pay later, so the forward must be higher than spot.

---

#### Why would $F = S$ allow arbitrage?

If $F = S$, but $r > 0$, then:

- Borrow money to buy the asset at $S$.
- Sell a forward at $F$.
- At maturity: deliver asset, repay loan with interest.

You earn a **riskless profit** — arbitrage.

So: under $q = 0$, it must hold that $F > S$ to prevent arbitrage.

---
---


In [ ]:
#mom.head()
mom_spx = mom[mom['ticker'] == 'SPX']
mom_spx.head()

In [ ]:
mom_clean = mom[
    ((mom['ticker'] != 'SPX') & (mom['var'] > 1e-7) & 
     (mom['skew'].abs() < 1000) & (mom['kurt'].abs() < 10000)) |
    (mom['ticker'] == 'SPX')
]


In [ ]:
mom_clean.head()


## 1.6 Interpretation of the SPX Daily Risk-Neutral Moments – February 2023

The graphs above show the the **first four risk-neutral moments** — mean ($\mu$), variance ($\sigma^2$), skewness, and kurtosis — of the return distribution for the SPX index throughout February 2023. These statistics are derived from out-of-the-money option prices following the methodology of Bakshi, Kapadia, and Madan (2003). They reflect market participants' expectations under the risk-neutral measure and provide valuable insight into perceived risks and uncertainties.

#### 1. Drift ($\mu$)
The drift measures the expected return of the underlying asset under the risk-neutral measure. It remains positive and relatively stable, fluctuating around 0.0024 to 0.0028. This suggests that option-implied forward prices modest positive growth expectations for the SPX over the next 30 days. Because this is computed under the risk-neutral measure, it reflects the market's valuation of future payoffs discounted at the risk-free rate. The absence of strong variation implies no major repricing of expected direction during the month.

#### 2. Variance ($\sigma^2$)
Implied variance represents the market’s forecast of volatility over the option horizon. We observe that variance increases sharply around February 10 and again near February 21, with values peaking above 0.0034. These spikes may correspond to anticipated macroeconomic events such as inflation releases, FOMC minutes, or earnings announcements. Variance is a key input for risk management and volatility trading strategies, and these movements reflect fluctuating uncertainty levels in the market.

#### 3. Skewness
Skewness captures the **asymmetry** of the implied return distribution. The SPX shows persistently **negative skewness** throughout February, ranging from roughly -2.3 to -2.8. This is economically intuitive: investors typically demand protection against downside moves, which inflates the price of OTM puts relative to calls. The increasing steepness around mid-February may indicate heightened crash concern or demand for tail hedging. Strong negative skew is a hallmark of equity index options and reflects risk aversion and insurance motives in the market.

#### 4. Kurtosis
Kurtosis measures the **fat-tailedness** or likelihood of extreme return realizations. The values are unusually high — consistently above 20 — with peaks around February 17 and 23. Elevated kurtosis implies the market assigns significant probability mass to extreme price changes in either direction. This may stem from uncertain policy signals, earnings season surprises, or broader macroeconomic risk. High kurtosis is especially relevant for strategies that are sensitive to tail risk, such as barrier options, portfolio stress tests, or risk parity allocations.

---

### Summary and Economic Significance
These moment estimates provide a **forward-looking view of the market’s beliefs** about return distributions. While traditional models often assume Gaussian returns, this analysis uncovers rich dynamics:

- **Drift and variance** reflect the baseline outlook and perceived uncertainty.
- **Skewness and kurtosis** encode investor fears about asymmetric losses and rare tail events.

Such insights are crucial for option pricing, asset allocation, and understanding risk premia. Moreover, these moments will serve as the foundation for the next tasks, where skewness is decomposed into systematic and idiosyncratic components to explore how market-wide and firm-specific risks are priced.


In [ ]:
spx = mom_clean[mom_clean['ticker'] == 'SPX']
spx_plot = spx.set_index('date')

for col, title in zip(['mu', 'var', 'skew', 'kurt'],
                      ['SPX – Drift (μ)', 'SPX – Varianz (σ²)', 'SPX – Schiefe', 'SPX – Kurtosis']):
    if spx_plot[col].dropna().empty:
        continue
    plt.figure()
    spx_plot[col].plot(title=title)
    plt.xlabel('Datum')
    plt.ylabel(col)
    plt.grid(True)
    plt.tight_layout()
    plt.show()


## 1.7 Visualization of the First Four Risk-Neutral Moments – First 10 Stocks

The plots above display the daily evolution of the first four risk-neutral moments for the first ten stocks (excluding SPX) in the dataset over February 2023. These moments are:

- **Drift ($\mu$):** Reflects the market-implied expected return under the risk-neutral measure. We observe differences in direction and stability across stocks. Some stocks (e.g., AAPL) show a more stable drift, while others (e.g., AAP) exhibit persistent negative or volatile drifts.

- **Variance ($\sigma^2$):** Captures the market's perceived uncertainty. Higher variance implies greater expected variability in returns. Notably, some stocks (e.g., AAP) display elevated and volatile implied variances throughout the month.

- **Skewness:** Measures asymmetry in the implied return distribution. Most stocks exhibit negative skewness, which is typical in equity markets due to downside risk. However, fluctuations and temporary positive skewness indicate shifts in market sentiment or firm-specific expectations.

- **Kurtosis:** Indicates the "tailedness" of the distribution. High kurtosis implies fat tails and greater probability of extreme outcomes. Some stocks (e.g., ABBV, ABT) show elevated kurtosis, suggesting market concern over potential extreme movements.

This analysis provides a detailed view of how investors price not only volatility but also asymmetry and tail risk for individual stocks, enabling richer risk management and portfolio insights.


In [ ]:
tickers = sorted([t for t in mom_clean['ticker'].unique() if t != 'SPX'])[:10]
data = {
    ticker: mom_clean[mom_clean['ticker'] == ticker].set_index('date')[['mu', 'var', 'skew', 'kurt']]
    for ticker in tickers
}

for col, title in zip(['mu', 'var', 'skew', 'kurt'],
                      ['Drift (μ)', 'Varianz (σ²)', 'Schiefe (Skewness)', 'Kurtosis']):
    plt.figure(figsize=(10, 5))
    for ticker in tickers:
        plt.plot(data[ticker].index, data[ticker][col], label=ticker)
    plt.title(title + ' – Erste 10 Aktien')
    plt.xlabel('Datum')
    plt.ylabel(col)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


---
---
## 2.0 Merging Implied Betas with Risk-Neutral Moments

In this section, we load and prepare the implied betas of individual S&P 500 constituents from the file `sp500_merged_implBeta_2023-02.parquet`. The implied beta measures the sensitivity of an individual stock's **risk-neutral return distribution** to that of the market (SPX), and is a key input for decomposing total skewness into systematic and idiosyncratic components.

More precisely, the **implied beta** reflects how strongly the option-implied return distribution of a stock co-moves with the market under the risk-neutral measure. Unlike traditional betas estimated from historical returns, implied betas are forward-looking and derived from option prices.

The following steps are performed:
- The dataset is renamed for consistency (`loctimestamp` to `date`, `Symbol` to `ticker`) and the `date` column is converted to datetime format.
- We then **merge** the implied beta data with the cleaned moments data (`mom_clean`) on both `date` and `ticker`, keeping only the matched rows (`inner` join).
- Additionally, we extract the SPX skewness for each date and merge it into the combined DataFrame under the column name `skew_mkt`.
- Finally, the column `implBeta` is renamed to `beta` for clarity.


In [ ]:
# Implizite Betas laden und vorbereiten
impl_betas = pq.read_table(ROOT / "sp500_merged_implBeta_2023-02.parquet").to_pandas()
impl_betas = impl_betas.rename(columns={
    "loctimestamp": "date",
    "Symbol": "ticker"
})
impl_betas["date"] = pd.to_datetime(impl_betas["date"])


In [ ]:
impl_betas.head()

In [ ]:
# Kombiniere mit Moments
df = mom_clean.merge(impl_betas, on=["date", "ticker"], how="inner")
spx_skew = mom_clean[mom_clean["ticker"] == "SPX"][["date", "skew"]].rename(columns={"skew": "skew_mkt"})
df = df.merge(spx_skew, on="date", how="left")

# Umbenennen für Klarheit
df.rename(columns={"implBeta": "beta"}, inplace=True)


## 2.1 Decomposition of Implied Skewness into Systematic and Idiosyncratic Components

To get a deeper insight into the sources of risk in individual stock returns, we decompose the **total implied skewness** into a **systematic (market-related)** and an **idiosyncratic (firm-specific)** component. This follows the methodology of BKM (2003), where the total skewness of a stock under the risk-neutral measure is attributed to its exposure to market skewness (systematic) and a residual part (idiosyncratic).

Let the third standardized moment (skewness) be given by $\text{skew}$ and the variance by $\sigma^2$. The **third central moment** is given by:
$$
c_3^{\text{total}} = \text{skew} \cdot \sigma^3
$$

Using the implied beta $\beta$ and the skewness of the market $\text{skew}_{\text{mkt}}$, we compute the **systematic component**:
$$
c_3^{\text{mkt}} = \beta^3 \cdot 3 \cdot \text{skew}_{\text{mkt}} \cdot \sigma^3
$$

The **idiosyncratic component** is then obtained as the residual:
$$
c_3^{\text{idio}} = c_3^{\text{total}} - c_3^{\text{mkt}}
$$

Finally, we convert these central moments back into **standardized skewness values**:
$$
\text{skew}_{\text{mkt,std}} = \frac{c_3^{\text{mkt}}}{\sigma^3}, \quad 
\text{skew}_{\text{idio,std}} = \frac{c_3^{\text{idio}}}{\sigma^3}
$$

This decomposition is very good for understanding if the skewness observed in option prices is driven primarily by market-wide risks or by firm-specific uncertainty. Such information is valuable in portfolio construction, risk management


In [ ]:
# 3. zentrale Momente rekonstruieren
df["c3_total"] = df["skew"] * (df["var"] ** 1.5)
df["c3_mkt"] = df["beta"] ** 3 * df["skew_mkt"] * (df["var"] ** 1.5)  # Marktanteil
df["c3_idio"] = df["c3_total"] - df["c3_mkt"]  # Idiosynkratisch

# Jetzt wieder standardisieren
df["skew_mkt_std"] = df["c3_mkt"] / (df["var"] ** 1.5)
df["skew_idio_std"] = df["c3_idio"] / (df["var"] ** 1.5)


In [ ]:
df[["date", "ticker", "skew", "skew_mkt_std", "skew_idio_std"]].head()


In [ ]:
example = "GS"
df_gs = df[df["ticker"] == example].sort_values("date")

plt.figure(figsize=(10, 5))
plt.plot(df_gs["date"], df_gs["skew"], label="total skew")
plt.plot(df_gs["date"], df_gs["skew_mkt_std"], "--", label="systematic skew")
plt.plot(df_gs["date"], df_gs["skew_idio_std"], "--", label="idiosyncratic skew")
plt.title(f"Decomposition of Implied Skewness, {example}")
plt.xlabel("Date")
plt.ylabel("Skewness")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 2.2 Example: Decomposition of Implied Skewness – Goldman Sachs (GS)

The plot above illustrates the decomposition of implied skewness for Goldman Sachs (GS) over February 2023. The solid blue line represents the **total implied skewness**, while the dashed orange and green lines represent the **systematic** and **idiosyncratic** components, respectively.

We observe that the idiosyncratic skewness is consistently higher (often positive), whereas the systematic skewness is negative throughout most of the sample. This suggests that the total skewness of GS is driven predominantly by firm-specific risk rather than market-wide effects. The market-related (systematic) skewness contributes only a small portion to the total skewness.

This decomposition helps us understand how much of the option-implied downside risk in GS is attributable to macroeconomic conditions versus company-specific factors. Such insights are useful for investors assessing tail risk and for structuring hedging strategies.


---
---
## 3.0 Comparing SPX Skewness to Average Stock Skewness

In this step, we compare the implied skewness of the SPX index with the average implied skewness of its constituent stocks on a daily basis. The goal is to assess whether the skewness of the market index is more pronounced than that of the individual stocks, as suggested in BKM (2003), Table 6.

We perform the following steps:
- Extract the daily skewness values for SPX and rename the column to \texttt{spx\_skew}.
- Filter out the individual stock data and compute the daily cross-sectional average of their implied skewness values, labeled \texttt{avg\_stock\_skew}.
- Merge both datasets on the date and compute the difference:  
  $$ \texttt{diff} = \texttt{spx\_skew} - \texttt{avg\_stock\_skew} $$

This difference quantifies how much more negatively skewed the index is compared to the average stock on a given day. A consistently negative difference would confirm the empirical observation that index options tend to exhibit more negative skewness than individual equity options, potentially due to market-wide crash risk being priced into index derivatives.


In [ ]:
# SPX-Skewness
spx_skew = mom_clean[mom_clean["ticker"] == "SPX"][["date", "skew"]].rename(columns={"skew": "spx_skew"})

# Einzelaktien-Skewness
stock_skews = mom_clean[mom_clean["ticker"] != "SPX"]

# Mittelwert über Aktien pro Tag
avg_stock_skew = stock_skews.groupby("date")["skew"].mean().reset_index().rename(columns={"skew": "avg_stock_skew"})

# Merge für den vergleich machen
skew_compare = spx_skew.merge(avg_stock_skew, on="date")


skew_compare["diff"] = skew_compare["spx_skew"] - skew_compare["avg_stock_skew"]


In [ ]:
skew_compare.head()


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(skew_compare["date"], skew_compare["spx_skew"], label="SPX Skewness", linewidth=2)
plt.plot(skew_compare["date"], skew_compare["avg_stock_skew"], label="Average Single Stock Skewness", linestyle="--")
plt.title("Implied Skewness: SPX vs. Avg Single Stocks (Feb 2023)")
plt.xlabel("Date")
plt.ylabel("Skewness")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 3.1 Economic Interpretation of the Plot
The plot above compares the implied skewness of the SPX index with the average implied skewness of its individual constituent stocks over the month of February 2023. This directly addresses Task 3, which refers to the finding by BKM (2003) that the skewness of the index tends to be more pronounced  than that of individual stocks.

From the graph, we observe that the SPX skewness (blue line) is consistently lower than the average single-stock skewness (orange dotted line). This means that the implied distribution of SPX returns is more left-skewed, indicating a greater market-perceived risk of large negative returns for the index compared to individual stocks.

Economically, this makes sense for several reasons. First, individual stock-specific risks are partially diversified in the index, leaving primarily systematic risk. Since investors are particularly concerned about market-wide crashes, they are willing to pay more for downside protection on the index, which increases the prices of out-of-the-money puts and leads to stronger negative skewness in index options.

Second, institutional investors often use SPX options to hedge portfolio risk. This persistent demand for downside protection inflates the implied volatility on the left tail of the SPX return distribution more than it does for individual stocks.

Lastly, idiosyncratic skewness in single stocks can cancel out in aggregation, making the average stock skew appear less extreme. Therefore, the stronger negative skewness observed in the SPX options reflects both aggregated market sentiment and the unique role of index options in hedging and speculation.

Overall, this analysis confirms the empirical observation made by BKM (2003) and highlights the importance of skewness as a measure of perceived tail risk in financial markets.
